# G2 · D5-TV2 · E0 Baseline

Frozen DINOv3 + Normal Feature Memory Bank, synthetic DEV AU-PRO@0.05.
Chọn **Runtime → Change runtime type → GPU** theo Colab Pro của bạn, sửa đường dẫn ở Cell 2 rồi **Run all**.
Seed run/memory = **42**; DEV seed = **17017** theo config D5 (E0 yêu cầu hai seed độc lập).
Notebook dùng CLI E0 có sẵn; không tính lại AU-PRO và không dùng TEST để tune.

**Trạng thái bản notebook trong repo: NOT RUN trên GPU Colab.** Chỉ nghiệm thu PASS khi full run có
8/8 category hợp lệ, đủ maps và đã export Drive. Smoke luôn nằm trong thư mục `smoke/` riêng.
Nếu phiên bị ngắt, Run all lại với cùng commit/config/đường dẫn; runner kiểm tra bank và skip run hợp lệ.
Bank chỉ resume sau khi được lưu hoàn chỉnh; ảnh TRAIN đang trích xuất dở sẽ cần chạy lại.

## Cell 1 — Environment
Pin code E0 và [official DINOv3](https://github.com/facebookresearch/dinov3/tree/6876159a11b4df116f30f667f8c9888617df0751).
Venv giữ CUDA PyTorch của Colab, cài requirements của repo; kernel notebook không cần restart.

In [ ]:
# @title Cell 1 — Environment
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import csv, hashlib, html, json, math, os, shlex, shutil, subprocess, sys, tarfile, threading
from IPython.display import HTML, Image as DisplayImage, display

REPO_URL = 'https://github.com/haduckien-raccoon/msila.git'
BRANCH = 'g2/member2'
MSILA_COMMIT = 'e3f79576d79a2a48b415be8d8949e8b4a795bd94'
DINO_COMMIT = '6876159a11b4df116f30f667f8c9888617df0751'
SEED = 42
LOCAL_WORK = Path('/content/msila_g2_e0')
PROJECT_DIR, DINO_DIR = LOCAL_WORK/'msila', LOCAL_WORK/'dinov3'
OUTPUT_ROOT = LOCAL_WORK/'outputs/G2/E0'
SESSION_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
ATTEMPT_DIR = OUTPUT_ROOT/'attempts'/SESSION_ID
LOCAL_WORK.mkdir(parents=True, exist_ok=True)
print('GPU evaluation: NOT RUN. Run seed:', SEED)
subprocess.run(['nvidia-smi'], check=True)

def checkout_pinned(url, destination, branch, commit):
    if not (destination/'.git').is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', branch, '--no-checkout', url, str(destination)], check=True)
    else:
        origin = subprocess.check_output(['git', '-C', str(destination), 'remote', 'get-url', 'origin'], text=True).strip()
        if origin != url or subprocess.check_output(['git', '-C', str(destination), 'status', '--porcelain'], text=True).strip():
            raise RuntimeError(f'BLOCKED: checkout khác source hoặc có thay đổi: {destination}')
    subprocess.run(['git', '-C', str(destination), 'fetch', '--depth', '1', 'origin', commit], check=True)
    subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    actual = subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
    if actual != commit:
        raise RuntimeError(f'BLOCKED: commit mismatch {actual} != {commit}')
    return actual

checkout_pinned(REPO_URL, PROJECT_DIR, BRANCH, MSILA_COMMIT)
checkout_pinned('https://github.com/facebookresearch/dinov3.git', DINO_DIR, 'main', DINO_COMMIT)
VENV_DIR = LOCAL_WORK/'venv'
if not (VENV_DIR/'bin/python').is_file():
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(VENV_DIR)], check=True)
PYTHON = str(VENV_DIR/'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '-r', str(PROJECT_DIR/'requirements.txt')], check=True)
RUN_ENV = dict(os.environ, PYTHONHASHSEED=str(SEED), PYTHONUNBUFFERED='1', MPLBACKEND='Agg')

def python_json(source):
    return json.loads(subprocess.check_output([PYTHON, '-c', source], cwd=PROJECT_DIR, env=RUN_ENV, text=True))

ENVIRONMENT = python_json("""
import json, platform, torch
from importlib.metadata import version
available = torch.cuda.is_available()
print(json.dumps(dict(python=platform.python_version(), torch=str(torch.__version__),
    cuda=torch.version.cuda, cuda_available=available,
    gpu_name=torch.cuda.get_device_name(0) if available else None,
    gpu_memory_bytes=torch.cuda.get_device_properties(0).total_memory if available else None,
    packages={name: version(name) for name in ('numpy','scipy','Pillow','PyYAML','matplotlib','safetensors')})))
""")
if not ENVIRONMENT['cuda_available']:
    raise RuntimeError('BLOCKED: CUDA không khả dụng; chọn GPU runtime rồi Run all lại.')
BASE_CONFIG = python_json("from pathlib import Path; import json, yaml; print(json.dumps(yaml.safe_load(Path('configs/g2_e0.yaml').read_text())))")
CATEGORIES = tuple(BASE_CONFIG['categories'])
DEV_SEED = BASE_CONFIG['evaluation']['dev_seed']
BACKBONE_NAME = BASE_CONFIG['backbone']['name']
print(json.dumps(dict(environment=ENVIRONMENT, branch=BRANCH, msila_commit=MSILA_COMMIT,
                     dino_commit=DINO_COMMIT, run_seed=SEED, dev_seed=DEV_SEED, backbone=BACKBONE_NAME), indent=2))
subprocess.run([PYTHON, 'scripts/run_g2_e0.py', '--help'], cwd=PROJECT_DIR, env=RUN_ENV, check=True)

## Cell 2 — Dataset & Checkpoint
Sửa `DRIVE_PROJECT` và `CHECKPOINT_ON_DRIVE` nếu cần. Mỗi pattern phải khớp đúng một archive/category.
**Copy đủ 8 `.tar.gz` Drive → Colab trước, sau đó mới giải nén local.**
Chỉ lấy TRAIN/good và VALIDATION/good, giữ native resolution. Synthetic DEV được tạo bởi
`G1NativeDataset` theo protocol sẵn có, không cần archive DEV riêng.
Quy trình copy archive rồi giải nén local cũng được nêu trong [Colab FAQ](https://research.google.com/colaboratory/faq.html).

In [ ]:
# @title Cell 2 — Dataset & Checkpoint
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PROJECT = '/content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9' # @param {type:'string'}
CHECKPOINT_ON_DRIVE = '' # @param {type:'string'}
RUN_TAG = 'D5_E0_seed42' # @param {type:'string'}
DRIVE_PROJECT = Path(DRIVE_PROJECT)
ARCHIVE_DIR = DRIVE_PROJECT/'data'
ARCHIVE_PATTERNS = {category: f'*{category}*.tar.gz' for category in CATEGORIES}
DRIVE_OUTPUT = DRIVE_PROJECT/'outputs/G2/E0'/RUN_TAG/MSILA_COMMIT[:12]
LOCAL_ARCHIVES, DATA_ROOT = LOCAL_WORK/'archives', LOCAL_WORK/'dataset'
CHECKPOINT_SOURCE = (Path(CHECKPOINT_ON_DRIVE) if CHECKPOINT_ON_DRIVE.strip() else
                     DRIVE_PROJECT/'weights'/Path(BASE_CONFIG['backbone']['checkpoints'][BACKBONE_NAME]).name)
if not CHECKPOINT_SOURCE.is_file():
    raise FileNotFoundError(f'BLOCKED: thiếu pretrained checkpoint: {CHECKPOINT_SOURCE}')
if not ARCHIVE_DIR.is_dir():
    raise FileNotFoundError(f'BLOCKED: thiếu thư mục 8 archive: {ARCHIVE_DIR}')

def copy_atomic(source, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name+'.tmp')
    shutil.copy2(source, temporary)
    temporary.replace(target)
    return target

def sha256_file(path):
    with Path(path).open('rb') as handle:
        digest = hashlib.sha256()
        for chunk in iter(lambda: handle.read(8*1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def selected_good_path(member, category):
    path = PurePosixPath(member.name)
    if path.is_absolute() or '..' in path.parts:
        raise ValueError(f'BLOCKED: archive path không hợp lệ: {member.name}')
    if not member.isfile():
        return None
    parts = path.parts
    split_indices = [i for i, part in enumerate(parts) if part.lower() in ('train', 'validation')]
    if len(split_indices) != 1:
        return None
    index = split_indices[0]
    if len(parts) <= index+2 or parts[index+1].lower() != 'good':
        return None
    if any(part in CATEGORIES and part != category for part in parts[:index]):
        raise ValueError(f'BLOCKED: archive {category} chứa category khác: {member.name}')
    return Path(category, parts[index].upper(), 'good', *parts[index+2:])

def prepare_archives(archive_dir, patterns, local_archives, data_root):
    selected = {}
    for category, pattern in patterns.items():
        candidates = sorted(path for path in archive_dir.glob(pattern) if path.is_file())
        if len(candidates) != 1:
            raise FileNotFoundError(f'BLOCKED: {category} cần đúng 1 archive ({pattern}), tìm thấy {candidates}')
        selected[category] = candidates[0]
    if len(selected) != 8 or len(set(selected.values())) != 8:
        raise ValueError('BLOCKED: cần 8 archive riêng biệt cho 8 category.')
    local_archives.mkdir(parents=True, exist_ok=True)
    data_root.mkdir(parents=True, exist_ok=True)
    if sum(path.stat().st_size for path in selected.values()) > shutil.disk_usage(local_archives).free:
        raise RuntimeError('BLOCKED: thiếu disk local để copy đủ 8 archive.')
    # Phase 1 finishes every Drive copy before any tarfile is opened.
    local = {}
    for category, source in selected.items():
        local[category] = copy_atomic(source, local_archives/source.name)
        print(f'COPIED {category}: {local[category]}', flush=True)
    # Inspect only local archives, budget disk for good images before extraction.
    plans, expanded = {}, 0
    for category, archive_path in local.items():
        with tarfile.open(archive_path, 'r:gz') as archive:
            plan = []
            for member in archive:
                relative = selected_good_path(member, category)
                if relative is not None:
                    plan.append((member.name, relative, member.size))
        if not plan or len({str(relative) for _, relative, _ in plan}) != len(plan):
            raise ValueError(f'BLOCKED: {category} thiếu good images hoặc có path trùng.')
        plans[category] = plan
        expanded += sum(size for _, _, size in plan)
    if expanded > shutil.disk_usage(data_root).free:
        raise RuntimeError(f'BLOCKED: thiếu disk local để giải nén {expanded/2**30:.2f} GiB good images.')
    manifest = []
    for category, archive_path in local.items():
        selected_names = {name: relative for name, relative, _ in plans[category]}
        with tarfile.open(archive_path, 'r:gz') as archive:
            for member in archive:
                if member.name in selected_names:
                    target = data_root/selected_names[member.name]
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with archive.extractfile(member) as source, target.open('wb') as destination:
                        shutil.copyfileobj(source, destination)
        manifest.append(dict(category=category, archive=archive_path.name,
                             sha256=sha256_file(archive_path), good_files=len(plans[category])))
        print(f'EXTRACTED {category}: {len(plans[category])} good files', flush=True)
    return manifest

archive_manifest = prepare_archives(ARCHIVE_DIR, ARCHIVE_PATTERNS, LOCAL_ARCHIVES, DATA_ROOT)
LOCAL_CHECKPOINT = copy_atomic(CHECKPOINT_SOURCE, LOCAL_WORK/'weights'/CHECKPOINT_SOURCE.name)
# Restore completed evidence before launching --resume; scientific identity is checked by E0.
if DRIVE_OUTPUT.is_dir():
    shutil.copytree(DRIVE_OUTPUT, OUTPUT_ROOT, dirs_exist_ok=True)
ATTEMPT_DIR.mkdir(parents=True, exist_ok=True)
COLAB_CONFIG = OUTPUT_ROOT/'run_config.yaml'
runtime_config = json.loads(json.dumps(BASE_CONFIG))
runtime_config['memory'].update(sampling_seed=SEED, device='cuda')
# Only execution device and the explicitly requested sampling seed change; DEV protocol/budgets stay D5.
subprocess.run([PYTHON, '-c', 'import json, sys, yaml; from pathlib import Path; Path(sys.argv[1]).write_text(yaml.safe_dump(json.load(sys.stdin), sort_keys=False))', str(COLAB_CONFIG)],
               input=json.dumps(runtime_config), text=True, check=True)
COMMON_ARGS = ['--config', str(COLAB_CONFIG), '--data-root', str(DATA_ROOT), '--repo-dir', str(DINO_DIR),
               '--weights', str(LOCAL_CHECKPOINT), '--output-root', str(OUTPUT_ROOT), '--device', 'cuda', '--save-maps', '--resume']
CONFIG_CODE = f'cfg = resolve_config(parse_args({COMMON_ARGS!r}))'
preflight = python_json("from src.eval.e0 import resolve_config, parse_args, asset_reasons, discover_sources; import json\n" + CONFIG_CODE + "\n"
    "reasons = asset_reasons(cfg, 'cuda')\n"
    "if reasons: raise RuntimeError('BLOCKED: ' + '; '.join(reasons))\n"
    "counts = {}\n"
    "for category in cfg['categories']:\n"
    "    pools, _ = discover_sources(cfg, category)\n"
    "    counts[category] = {role: len(records) for role, records in pools.items()}\n"
    "print(json.dumps(dict(counts=counts, sampling_seed=cfg['memory']['sampling_seed'], dev_seed=cfg['evaluation']['dev_seed'])))")
provenance = dict(environment=ENVIRONMENT, git_commit=MSILA_COMMIT, branch=BRANCH, dino_commit=DINO_COMMIT,
                  run_seed=SEED, dev_seed=DEV_SEED, checkpoint_sha256=sha256_file(LOCAL_CHECKPOINT),
                  archives=archive_manifest, preflight=preflight, session_id=SESSION_ID)
(ATTEMPT_DIR/'provenance.json').write_text(json.dumps(provenance, indent=2))
shutil.copy2(COLAB_CONFIG, ATTEMPT_DIR/'run_config.yaml')
print(json.dumps(preflight, indent=2))
print('Output Drive:', DRIVE_OUTPUT)

## Cell 3 — Preflight / Smoke Test
Một category, bank 32 vector, một nguồn TRAIN, một nguồn DEV × ba anomaly variants theo `--smoke`.
Map/GT, QA, bank và AU-PRO đều do CLI E0 tạo. Backup Drive định kỳ 60 giây và sau mỗi stage.

In [ ]:
# @title Cell 3 — Preflight / Smoke Test
SYNC_LOCK, SYNCED = threading.Lock(), {}

def sync_outputs(force=False):
    with SYNC_LOCK:
        DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
        for source in sorted(OUTPUT_ROOT.rglob('*')):
            if not source.is_file() or source.name.endswith('.tmp'):
                continue
            relative = source.relative_to(OUTPUT_ROOT)
            stat = source.stat()
            fingerprint = (stat.st_size, stat.st_mtime_ns)
            if force or SYNCED.get(str(relative)) != fingerprint:
                copy_atomic(source, DRIVE_OUTPUT/relative)
                after = source.stat()
                if (after.st_size, after.st_mtime_ns) == fingerprint:
                    SYNCED[str(relative)] = fingerprint

def run_logged(arguments, stage):
    command = [PYTHON, 'scripts/run_g2_e0.py', *arguments]
    log_path = ATTEMPT_DIR/f'{stage}.log'
    stop = threading.Event()
    def backup():
        while not stop.wait(60):
            try:
                sync_outputs()
            except OSError as error:
                print(f'BACKUP BLOCKED: {error}', flush=True)
    worker = threading.Thread(target=backup, daemon=True)
    print(shlex.join(command), flush=True)
    worker.start()
    process = None
    try:
        with log_path.open('w') as log:
            log.write(shlex.join(command)+'\n')
            log.flush()
            process = subprocess.Popen(command, cwd=PROJECT_DIR, env=RUN_ENV, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            return_code = process.wait()
        (ATTEMPT_DIR/f'{stage}_command.json').write_text(json.dumps(dict(command=command, return_code=return_code), indent=2))
        return return_code
    finally:
        if process is not None and process.poll() is None:
            process.terminate()
            process.wait()
        stop.set()
        worker.join()
        sync_outputs()

def inspect_results(root, categories, mode, return_code):
    issues, valid, rows = [], [], []
    summary_path = root/'summary.json'
    summary = json.loads(summary_path.read_text()) if summary_path.is_file() else {}
    csv_path = root/'e0_metrics_8categories.csv'
    csv_rows = list(csv.DictReader(csv_path.open())) if csv_path.is_file() else []
    if len(csv_rows) != 8 or {row['category'] for row in csv_rows} != set(CATEGORIES):
        issues.append('CSV không chứa đúng 8 category.')
    indexed = {row['category']: row for row in csv_rows}
    reported = {row['category']: row for row in summary.get('categories', [])}
    for category in categories:
        folder = root/category
        path = folder/'metrics.json'
        metric = json.loads(path.read_text()) if path.is_file() else {}
        value = metric.get('synthetic_dev_aupro_0_05')
        checks = [metric.get('status') == 'PASS', reported.get(category, {}).get('status') == 'PASS',
                  metric.get('category') == category, metric.get('mode') == mode,
                  metric.get('verification_scope') == 'pretrained_dinov3',
                  metric.get('split') == 'dev_synthetic', metric.get('aupro_max_fpr') == .05,
                  metric.get('frozen_backbone_unchanged') is True, metric.get('trainable_parameters') == 0,
                  metric.get('qa_status') == 'PASS', metric.get('native_resolution') is True,
                  type(value) in (int, float) and math.isfinite(value) and 0 <= value <= 1,
                  (folder/'normal_memory.pt').is_file(), (folder/'normal_memory.sha256.json').is_file(),
                  (folder/'qa_report.json').is_file(), (folder/'dev_manifest.json').is_file(),
                  len(list((folder/'maps').glob('*/anomaly_map.npy'))) == metric.get('n_samples', -1),
                  len(list((folder/'maps').glob('*/gt_mask.npy'))) == metric.get('n_samples', -1)]
        try:
            checks.append(float(indexed[category]['synthetic_dev_aupro_0_05']) == value)
        except (ValueError, KeyError):
            checks.append(False)
        if all(checks):
            valid.append(category)
        else:
            issues.append(f'{category}: metric/provenance/maps thiếu hoặc không hợp lệ ({metric.get("status", "NOT RUN")}).')
        rows.append(dict(category=category, status=metric.get('status', 'NOT RUN'),
                         au_pro_0_05=value if type(value) in (int, float) and math.isfinite(value) else None,
                         valid=all(checks)))
    expected_status = 'PASS' if mode == 'full' else 'SMOKE PASS'
    if return_code != 0 or summary.get('status') != expected_status:
        issues.append(f'CLI/summary: return_code={return_code}, status={summary.get("status", "NOT RUN")}.')
    macro = summary.get('macro_synthetic_dev_aupro_0_05')
    if mode == 'full' and (summary.get('real_categories_pass') != 8 or len(categories) != 8
                           or type(macro) not in (int, float) or not math.isfinite(macro) or not 0 <= macro <= 1):
        issues.append('Full acceptance cần 8/8 kết quả thật và macro hữu hạn từ evaluator.')
    status = (expected_status if not issues else 'BLOCKED' if summary.get('status') == 'BLOCKED' else
              'NOT RUN' if return_code is None or summary.get('status') == 'NOT RUN' else 'FAIL')
    return dict(status=status, valid_categories=len(valid), expected_categories=len(categories),
                real_categories_pass=len(valid) if mode == 'full' else 0,
                macro_au_pro_0_05=macro if status == 'PASS' else None, issues=issues, rows=rows)

SMOKE_CATEGORY = 'rice'
SMOKE_CODE = run_logged([*COMMON_ARGS, '--categories', SMOKE_CATEGORY, '--smoke'], 'smoke')
SMOKE_REPORT = inspect_results(OUTPUT_ROOT/'smoke', [SMOKE_CATEGORY], 'smoke', SMOKE_CODE)
(ATTEMPT_DIR/'smoke_check.json').write_text(json.dumps(SMOKE_REPORT, indent=2, allow_nan=False))
sync_outputs()
print(json.dumps(SMOKE_REPORT, indent=2))
if SMOKE_REPORT['status'] != 'SMOKE PASS':
    raise RuntimeError('BLOCKED: smoke chưa hợp lệ; chưa chạy full evaluation. Log đã backup Drive.')
print('Smoke PASS; full nghiệm thu vẫn NOT RUN.')

## Cell 4 — Full Evaluation
CLI chạy đủ 8 category với cùng config D5 và seed sampling 42.
Không có `--smoke`/giảm budget; `--resume` xác minh identity và skip kết quả hợp lệ.
Nếu cần đổi config của một run đã hoàn thành, dùng RUN_TAG mới; không ghi đè bank khác identity.

In [ ]:
# @title Cell 4 — Full Evaluation
if SMOKE_REPORT['status'] != 'SMOKE PASS':
    raise RuntimeError('BLOCKED: cần smoke hợp lệ trước full evaluation.')
FULL_CODE = run_logged([*COMMON_ARGS, '--categories', 'all'], 'full')
print('Full CLI return code:', FULL_CODE, '— nghiệm thu ở Cell 5, export ở Cell 6.')

## Cell 5 — Results & Visualization
Đọc AU-PRO/macro từ artifacts của evaluator; không tính metric mới.
Ba ví dụ dùng chính `G1NativeDataset` để khôi phục synthetic Input với DEV seed/config đã lưu,
kiểm tra exact mask khớp file GT, rồi vẽ map native được CLI lưu với `--save-maps`.

In [ ]:
# @title Cell 5 — Results & Visualization
ACCEPTANCE = inspect_results(OUTPUT_ROOT, CATEGORIES, 'full', globals().get('FULL_CODE'))
table_rows = ''.join(f'<tr><td>{html.escape(row["category"])}</td><td>{html.escape(row["status"])}</td>'
                     f'<td>{row["au_pro_0_05"] if row["au_pro_0_05"] is not None else "—"}</td>'
                     f'<td>{row["valid"]}</td></tr>' for row in ACCEPTANCE['rows'])
display(HTML('<table><tr><th>Category</th><th>Run status</th><th>AU-PRO@0.05</th><th>Valid</th></tr>'+table_rows+'</table>'))
print(f"{ACCEPTANCE['status']} — valid {ACCEPTANCE['valid_categories']}/8; macro = {ACCEPTANCE['macro_au_pro_0_05']}")

# Rendering uses the existing native synthetic dataset; no new anomaly or metric implementation.
VISUALIZATION_CODE = r"""
import json, numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from src.data.loader import G1NativeDataset
from src.eval.e0 import parse_args, resolve_config, discover_sources
""" + CONFIG_CODE + "\n" + f"eligible = {[row['category'] for row in ACCEPTANCE['rows'] if row['valid']]!r}\n" + r"""
output = Path(cfg['output_root'])
rendered = []
for category in cfg['categories']:
    if category not in eligible: continue
    folder = output/category
    metric_path = folder/'metrics.json'
    if not metric_path.is_file(): continue
    metric = json.loads(metric_path.read_text())
    if metric.get('status') != 'PASS' or metric.get('mode') != 'full': continue
    if metric.get('verification_scope') != 'pretrained_dinov3': continue
    manifest = json.loads((folder/'dev_manifest.json').read_text())['samples']
    index = next(i for i, meta in enumerate(manifest) if meta['synthetic']['is_anomaly'])
    pools, _ = discover_sources(cfg, category)
    native = G1NativeDataset(pools['dev'], cfg['synthetic_protocol'], seed=cfg['evaluation']['dev_seed'],
                             role='dev', variants=cfg['data']['dev_variants_per_image'], fixed=True)
    sample = native[index]
    if sample['meta'] != manifest[index]: raise ValueError('DEV metadata changed')
    example = folder/'maps'/sample['meta']['sample_id']
    score, mask = np.load(example/'anomaly_map.npy'), np.load(example/'gt_mask.npy')
    if not np.array_equal(mask, sample['mask'][0].numpy()): raise ValueError('DEV mask mismatch')
    image = sample['image'].permute(1, 2, 0).numpy()
    if image.shape[:2] != score.shape or mask.shape != score.shape or not np.isfinite(score).all():
        raise ValueError('Invalid native visualization maps')
    Image.fromarray(np.rint(image.clip(0,1)*255).astype(np.uint8)).save(example/'input.png')
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image); axes[0].set_title('Synthetic DEV input')
    axes[1].imshow(mask, cmap='gray', vmin=0, vmax=1); axes[1].set_title('Exact ground truth mask')
    heatmap = axes[2].imshow(score, cmap='inferno', vmin=0, vmax=1)
    axes[2].set_title('Native E0 anomaly map'); fig.colorbar(heatmap, ax=axes[2], fraction=.046)
    for ax in axes: ax.axis('off')
    fig.suptitle(f'{category} · AU-PRO@0.05 = {metric["synthetic_dev_aupro_0_05"]:.6f}')
    fig.tight_layout()
    target = output/'visualizations'/f'{category}.png'; target.parent.mkdir(exist_ok=True)
    fig.savefig(target, dpi=120); plt.close(fig); rendered.append(str(target))
    if len(rendered) == 3: break
print(json.dumps(rendered))
"""
try:
    for filename in python_json(VISUALIZATION_CODE):
        display(DisplayImage(filename=filename))
    if ACCEPTANCE['status'] == 'PASS' and not list((OUTPUT_ROOT/'visualizations').glob('*.png')):
        raise RuntimeError('Thiếu visualization cho nghiệm thu.')
except Exception as error:
    if ACCEPTANCE['status'] == 'PASS':
        ACCEPTANCE['status'] = 'FAIL'
    ACCEPTANCE['issues'].append(f'Visualization: {error}')
(ATTEMPT_DIR/'acceptance.json').write_text(json.dumps(ACCEPTANCE, indent=2, allow_nan=False))
sync_outputs()
print(json.dumps({key: value for key, value in ACCEPTANCE.items() if key != 'rows'}, indent=2))

## Cell 6 — Export
Backup toàn bộ output E0: CSV, 8 metrics JSON, summary, normal banks/checksums, QA/DEV manifests,
native maps/GT, ví dụ Input, hình so sánh và logs. Mỗi attempt lưu config, Git commit, archive/checkpoint
SHA256, seed, CUDA/GPU/PyTorch và pip freeze. PASS export chỉ sau khi toàn bộ file đồng bộ.

In [ ]:
# @title Cell 6 — Export
with (ATTEMPT_DIR/'pip_freeze.txt').open('w') as handle:
    subprocess.run([PYTHON, '-m', 'pip', 'freeze'], stdout=handle, check=True)
(ATTEMPT_DIR/'nvidia_smi.txt').write_text(subprocess.check_output(['nvidia-smi'], text=True))
ACCEPTANCE = dict(ACCEPTANCE)
ACCEPTANCE['export_status'] = 'NOT RUN'
try:
    sync_outputs(force=True)
    ACCEPTANCE['export_status'] = 'PASS'
    (ATTEMPT_DIR/'acceptance.json').write_text(json.dumps(ACCEPTANCE, indent=2, allow_nan=False))
    copy_atomic(ATTEMPT_DIR/'acceptance.json', DRIVE_OUTPUT/'attempts'/SESSION_ID/'acceptance.json')
except OSError as error:
    ACCEPTANCE['status'], ACCEPTANCE['export_status'] = 'BLOCKED', 'BLOCKED'
    ACCEPTANCE['issues'].append(f'Drive export: {error}')
(ATTEMPT_DIR/'acceptance.json').write_text(json.dumps(ACCEPTANCE, indent=2, allow_nan=False))
print(f"{ACCEPTANCE['status']} — coverage {ACCEPTANCE['valid_categories']}/8, export {ACCEPTANCE['export_status']}")
print('CSV:', DRIVE_OUTPUT/'e0_metrics_8categories.csv')
print('Summary:', DRIVE_OUTPUT/'summary.json')
print('Config / environment / seed / logs:', DRIVE_OUTPUT/'attempts'/SESSION_ID)
if ACCEPTANCE['issues']:
    print('\n'.join(ACCEPTANCE['issues']))